# How to use a custom DIAL adapter

An [adapter](https://github.com/epam/ai-dial-sdk) is the service that implements a DIAL model: it accepts Unified API requests from DIAL Core, translates them into a provider's native API, and translates the answer back.

This notebook calls `mock-model`, a deployment served by the worked example in `dial-samples/custom-adapter`. Behind it sits a stand-in provider with a deliberately non-OpenAI API, running as two upstreams — `provider-eu` and `provider-us` — each with its own API key.

Along the way it shows the two things DIAL Core does for a model that an application does not get: it load-balances across `upstreams`, and it authenticates to each upstream separately.

In [ ]:
!pip install -q requests==2.32.3

In [ ]:
import os
import time
from collections import Counter

import requests

**Step 1**: point `DIAL_URL` at a running DIAL Core. The provider is a stand-in, so no provider account is needed.

In [ ]:
DIAL_URL = os.environ.get("DIAL_URL", "http://localhost:8080")
API_KEY = "dial_api_key"
HEADERS = {"Api-Key": API_KEY, "Content-Type": "application/json"}
SEPARATOR = "--- served by ---"


def ask(deployment, text, attempts=5):
    """Call a deployment, retrying only on 5xx.

    DIAL Core's local development storage can emit a transient 5xx under load.
    Retrying those keeps the notebook stable without softening any assertion:
    a genuinely broken adapter fails every attempt and raises below, and a 4xx
    (such as a rejected upstream key) is returned as-is for the caller to check.
    """
    for _ in range(attempts):
        reply = requests.post(
            f"{DIAL_URL}/openai/deployments/{deployment}/chat/completions",
            headers=HEADERS,
            json={"messages": [{"role": "user", "content": text}]},
            timeout=60,
        )
        if reply.status_code < 500:
            return reply
        time.sleep(1)

    raise RuntimeError(
        f"{deployment} returned 5xx {attempts} times; "
        f"last was {reply.status_code}: {reply.text}"
    )

**Step 2**: call the model through the adapter.

You send an ordinary Unified API request. The provider behind it speaks a different protocol entirely — it takes `{"prompt": ..., "max_length": ...}` and answers with `{"generated_text": ..., "tokens_used": ...}`. Translating that is the adapter's whole job.

In [ ]:
response = ask("mock-model", "Hello, can you summarize the weather today?")
response.raise_for_status()

content = response.json()["choices"][0]["message"]["content"]
print(content)

The reply arrives in Unified API shape, so the translation worked in both directions. The adapter also appends which upstream served the request, which the next step uses.

In [ ]:
assert "Mock response to:" in content, "the provider's text did not survive translation"
assert SEPARATOR in content, "the adapter did not report the upstream"

print("the adapter translated the provider's format into the Unified API")

**Step 3**: watch DIAL Core load-balance across upstreams.

The model declares two `upstreams` with equal weight. For each request Core picks one and tells the adapter which, in the `X-UPSTREAM-ENDPOINT` header. Over enough requests both should appear — the adapter itself does no balancing.

In [ ]:
served_by = Counter()

for _ in range(20):
    reply = ask("mock-model", "hi")
    reply.raise_for_status()
    text = reply.json()["choices"][0]["message"]["content"]
    served_by[text.split(SEPARATOR, 1)[1].strip()] += 1

print(dict(served_by))

assert served_by["provider-eu"] > 0, "provider-eu never served a request"
assert served_by["provider-us"] > 0, "provider-us never served a request"

print("DIAL Core balanced the requests across both upstreams")

**Step 4**: confirm the API key comes from DIAL Core, not from the adapter.

Each upstream has a different key, and each provider instance rejects anything else. Since every call above succeeded, the adapter must have sent the key belonging to whichever upstream Core routed it to — a key baked into the adapter image could only ever satisfy one of them.

`mock-model-bad-key` proves the check is live: it is the same adapter and the same provider, with a wrong key in the DIAL Core configuration.

In [ ]:
bad = ask("mock-model-bad-key", "hi")

assert bad.status_code == 401, (
    f"expected the provider to reject the key with 401, got "
    f"{bad.status_code}: {bad.text}"
)
# Check it failed for the right reason, not by falling over somewhere else.
assert "rejected the API key" in bad.text, (
    f"expected the provider to reject the key, got: {bad.text}"
)

print(f"wrong key rejected (HTTP {bad.status_code})")
print("the key travels per request from DIAL Core configuration")

## Next steps

- The adapter source, and a standalone Docker Compose stack you can run on its own, are in [`dial-samples/custom-adapter`](../../dial-samples/custom-adapter).
- To build one yourself, follow [Tutorial: custom adapter](https://docs.dialx.ai/v2/building-with-dial/adapters/tutorial-custom-adapter).
- For the full `upstreams` schema — weights, tiers, retries — see the [config.json models reference](https://docs.dialx.ai/v2/operating-dial/configuration/core/config-json/models).